# Silver Layer — ICU outputevents

## READINESS: partly profiled, rules drafted, awaiting remaining profiling and team approval

The type, missingness, flag, range and relationship counts recorded in **Profiling
Findings** come from the run of 2026-10-19/20 (saved outputs of the previous version of this
notebook, which published `cdr_mimiciv.silver.outputevents` with the same two DQ flags).
Checks marked *not yet profiled* must be run and recorded before approval. Until the
`profiling_approved` widget is set to `true`, the publish step raises an error and nothing
is written. The dependency-ordered runner excludes this notebook by default.

## Source
`<source_schema>.outputevents` (default `cdr_mimiciv.bronze.outputevents`)

## Target
`<target_schema>.outputevents` (default `cdr_mimiciv.silver.outputevents`), published with
CREATE OR REPLACE semantics. No tables are dropped.

## Grain (official MIMIC-IV icu documentation, `outputevents`)
One row per recorded patient output (e.g. urine, drain) during an ICU stay (source:
MetaVision). The documentation defines **no unique row identifier**. Documented fields:
`subject_id`, `hadm_id`, `stay_id` (patient, hospital stay and ICU stay identifiers; links
to patients, admissions and icustays), `caregiver_id` (the caregiver who documented the
data), `charttime` (the time of an output event), `storetime` (time the observation was
manually input or manually validated by clinical staff), `itemid` (measurement type; links
to `d_items`), `value` and `valueuom` (the amount of a substance at `charttime`; the exact
start time is unknown, "but usually up to an hour before").
The documentation does not state NOT NULL constraints for any column, and does not state
whether negative values are valid.
[DOC] The uploaded documentation (v2.2) lists 4,234,967 rows; Bronze holds 5,359,395,
consistent with a later MIMIC-IV release.

## Silver transformation (derived from the profiling findings)
Principle, as in admissions, labevents and chartevents: preserve source value, identify
anomaly, add a derived column or DQ flag, let downstream users decide.
- Select every Bronze column unchanged, in its Bronze type and order. (The previous version
  re-cast every column through `TRY_CAST(TRIM(CAST(... AS STRING)))`; Bronze is already
  typed and 0 cast failures were profiled, so the re-cast is removed. Output types are
  unchanged.)
- Add DQ flags (our engineering choice), both non-null booleans, unchanged from the
  published table:
  - `dq_negative_value`: `value < 0` (604 rows profiled). Informational: the documentation
    neither allows nor forbids negative output values; rows are kept unchanged.
  - `dq_storetime_before_charttime`: `storetime < charttime` (757,757 rows profiled).
    Informational, not asserted to be an error: `value` is documented as the amount at
    `charttime`, so a round charting time can follow the validation time. Same name and
    meaning as labevents and chartevents.
- Add `_silver_processed_at`.
- No deduplication, no NULL filling, no timestamp correction, no unit conversion, no
  magnitude capping, no row removal. Per-item plausibility limits are decided after the
  item-level profile below; output totals and rates are Gold features.

## Open profiling questions
1. Are there exact duplicate rows? (No documented key.)
2. Values of `valueuom` (case, whitespace or empty-string variants).
3. Size of the `storetime` before `charttime` gap.
4. Which items carry negative values and extreme magnitudes (profiled maximum 4,323,410)?
5. Relationships (observational only): `(stay_id, subject_id, hadm_id)` in icustays;
   `itemid` in `d_items` with `linksto = 'outputevents'`; `caregiver_id` in caregiver;
   `charttime` outside the ICU stay `intime`/`outtime` window.

## Cost note
About 5.4 million rows. Profiling uses one aggregate pass, relationship checks broadcast the
small reference tables, no `exceptAll` is used, and the published table is checked with one
aggregate pass.

In [0]:
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")
dbutils.widgets.text("profiling_approved", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"
PROFILING_APPROVED = dbutils.widgets.get("profiling_approved").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

from pyspark.sql import functions as F

SOURCE_TABLE = f"{BRONZE}.outputevents"
TARGET_TABLE = f"{SILVER}.outputevents"

DOCUMENTED_COLUMNS = [
    "subject_id", "hadm_id", "stay_id", "caregiver_id", "charttime", "storetime",
    "itemid", "value", "valueuom"
]

In [0]:
outputevents_bronze = spark.table(SOURCE_TABLE)
SOURCE_COLUMNS = outputevents_bronze.columns

outputevents_bronze.printSchema()
print("Documented columns missing from Bronze:", [c for c in DOCUMENTED_COLUMNS if c not in SOURCE_COLUMNS])
print("Bronze columns not in documentation:", [c for c in SOURCE_COLUMNS if c not in DOCUMENTED_COLUMNS])
print("Target exists:", spark.catalog.tableExists(TARGET_TABLE))
display(outputevents_bronze.limit(10))

root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- stay_id: integer (nullable = true)
 |-- caregiver_id: integer (nullable = true)
 |-- charttime: timestamp (nullable = true)
 |-- storetime: timestamp (nullable = true)
 |-- itemid: integer (nullable = true)
 |-- value: double (nullable = true)
 |-- valueuom: string (nullable = true)

Documented columns missing from Bronze: []
Bronze columns not in documentation: []
Target exists: True


subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valueuom
10000032,29079034,39553978,18704,2180-07-23T15:00:00.000Z,2180-07-23T16:00:00.000Z,226560,175.0,mL
10000690,25860671,37081114,8787,2150-11-06T08:00:00.000Z,2150-11-06T09:08:00.000Z,226559,15.0,mL
10000690,25860671,37081114,8787,2150-11-06T09:00:00.000Z,2150-11-06T09:08:00.000Z,226559,15.0,mL
10000690,25860671,37081114,8787,2150-11-06T10:00:00.000Z,2150-11-06T09:50:00.000Z,226559,12.0,mL
10000690,25860671,37081114,8787,2150-11-06T11:00:00.000Z,2150-11-06T13:42:00.000Z,226559,15.0,mL
10000690,25860671,37081114,8787,2150-11-06T12:00:00.000Z,2150-11-06T13:42:00.000Z,226559,15.0,mL
10000690,25860671,37081114,9137,2150-11-06T13:00:00.000Z,2150-11-06T16:18:00.000Z,226559,25.0,mL
10000690,25860671,37081114,9137,2150-11-06T14:00:00.000Z,2150-11-06T16:18:00.000Z,226559,30.0,mL
10000690,25860671,37081114,9137,2150-11-06T15:00:00.000Z,2150-11-06T16:18:00.000Z,226559,25.0,mL
10000690,25860671,37081114,9137,2150-11-06T16:00:00.000Z,2150-11-06T16:18:00.000Z,226559,25.0,mL


In [0]:
# single-pass profile: missingness of every column, value checks, ranges and timestamp order
def _cnt(cond):
    return F.sum(F.when(cond, 1).otherwise(0))

gap_seconds = F.unix_timestamp("charttime") - F.unix_timestamp("storetime")

profile_exprs = [F.count("*").alias("total_rows")]
profile_exprs += [_cnt(F.col(c).isNull()).alias(f"null_{c}") for c in SOURCE_COLUMNS]
profile_exprs += [
    _cnt(F.col("value") < 0).alias("negative_value"),
    _cnt(F.col("value") == 0).alias("zero_value"),
    _cnt(F.isnan("value")).alias("nan_value"),
    _cnt(F.trim(F.col("valueuom")) == "").alias("valueuom_empty_string"),
    _cnt(F.col("storetime") < F.col("charttime")).alias("storetime_before_charttime"),
    _cnt(gap_seconds > 3600).alias("storetime_before_charttime_over_1h"),
    _cnt(gap_seconds > 86400).alias("storetime_before_charttime_over_24h"),
    _cnt(F.col("storetime") == F.col("charttime")).alias("storetime_equals_charttime"),
]
for c in ("charttime", "storetime", "value"):
    profile_exprs += [F.min(c).alias(f"min_{c}"), F.max(c).alias(f"max_{c}")]

out_profile = outputevents_bronze.agg(*profile_exprs).first().asDict()
for k, v in out_profile.items():
    print(f"{k}: {v}")

total_rows: 5359395
null_subject_id: 0
null_hadm_id: 0
null_stay_id: 0
null_caregiver_id: 0
null_charttime: 0
null_storetime: 0
null_itemid: 0
null_value: 0
null_valueuom: 0
negative_value: 604
zero_value: 232868
nan_value: 0
valueuom_empty_string: 0
storetime_before_charttime: 757757
storetime_before_charttime_over_1h: 1229
storetime_before_charttime_over_24h: 25
storetime_equals_charttime: 546498
min_charttime: 2110-01-11 12:30:00
max_charttime: 2214-07-26 14:00:00
min_storetime: 2110-01-11 12:50:00
max_storetime: 2214-07-26 14:39:00
min_value: -3765.0
max_value: 4323410.0


In [0]:
# exact duplicate rows: no documented key, so the only grain check is on all columns
duplicate_groups = (
    outputevents_bronze.groupBy(*SOURCE_COLUMNS).count().filter(F.col("count") > 1)
)
dup_summary = duplicate_groups.agg(
    F.count("*").alias("duplicate_groups"),
    F.sum(F.col("count") - 1).alias("surplus_rows"),
).first().asDict()
print("Exact duplicates:", dup_summary)
if dup_summary["duplicate_groups"]:
    display(duplicate_groups.orderBy(F.desc("count")).limit(50))

Exact duplicates: {'duplicate_groups': 0, 'surplus_rows': None}


In [0]:
# categorical values (small result set)
print("=== valueuom ===")
display(outputevents_bronze.groupBy("valueuom").count().orderBy(F.desc("count")))

=== valueuom ===


valueuom,count
mL,5359395


In [0]:
# item-level profile: negatives, magnitude and timestamp order per itemid (inspect, do not remove)
item_profile = (
    outputevents_bronze.groupBy("itemid")
    .agg(
        F.count("*").alias("rows"),
        _cnt(F.col("value") < 0).alias("negative_values"),
        F.min("value").alias("min_value"),
        F.expr("percentile_approx(value, 0.5)").alias("median_value"),
        F.expr("percentile_approx(value, 0.999)").alias("p999_value"),
        F.max("value").alias("max_value"),
        _cnt(F.col("storetime") < F.col("charttime")).alias("storetime_before_charttime"),
    )
    .join(F.broadcast(spark.table(f"{SILVER}.d_items").select("itemid", "label", "unitname")), "itemid", "left")
)

print("=== items with negative values ===")
display(item_profile.filter(F.col("negative_values") > 0).orderBy(F.desc("negative_values")))

print("=== items by maximum value ===")
display(item_profile.orderBy(F.desc("max_value")).limit(20))

=== items with negative values ===


itemid,rows,negative_values,min_value,median_value,p999_value,max_value,storetime_before_charttime,label,unitname
226588,344639,212,-1000.0,20.0,1300.0,6000.0,61224,Chest Tube #1,mL
226589,48105,143,-90.0,20.0,1200.0,2500.0,8315,Chest Tube #2,mL
226599,78738,27,-20.0,35.0,945.0,6500.0,11506,Jackson Pratt #1,mL
229414,1095,25,-90.0,10.0,960.0,1540.0,198,Chest Tube #4,mL
229413,3376,23,-120.0,10.0,600.0,1220.0,658,Chest Tube #3,mL
226579,19417,22,-800.0,200.0,2400.0,4000.0,1158,Stool,mL
226580,22519,21,-1000.0,300.0,1600.0,3600.0,1994,Fecal Bag,mL
226559,3599702,19,-3765.0,80.0,1300.0,30331.0,553410,Foley,mL
226583,11175,17,-1000.0,250.0,2000.0,4323410.0,1145,Rectal Tube,mL
226619,8659,15,-150.0,45.0,2100.0,7500.0,1152,Pigtail #1,mL


=== items by maximum value ===


itemid,rows,negative_values,min_value,median_value,p999_value,max_value,storetime_before_charttime,label,unitname
226583,11175,17,-1000.0,250.0,2000.0,4323410.0,1145,Rectal Tube,mL
226575,39137,5,-50.0,100.0,2000.0,1098560.0,5762,Nasogastric,mL
226560,386902,3,-1.0,250.0,1300.0,876587.0,21527,Void,mL
226633,14852,2,-1000.0,400.0,8570.0,129385.0,405,Pre-Admission,mL
226626,13460,0,0.0,150.0,23000.0,70000.0,57,OR EBL,mL
226559,3599702,19,-3765.0,80.0,1300.0,30331.0,553410,Foley,mL
226629,600,0,0.0,100.0,29000.0,29000.0,6,PACU EBL,mL
226625,1443,0,0.0,1400.0,11400.0,27000.0,41,Tap,mL
226632,1384,0,0.0,400.0,4700.0,14000.0,27,Cath Lab,mL
226617,1507,0,0.0,100.0,3000.0,14000.0,285,Sump #1,mL


In [0]:
# relationships (observational only; findings are recorded, not turned into flags here)
icustays_table = f"{SILVER}.icustays"
d_items_table = f"{SILVER}.d_items"
caregiver_table = f"{SILVER}.caregiver"

for _t in (icustays_table, d_items_table, caregiver_table):
    assert spark.catalog.tableExists(_t), f"Missing reference: {_t}"

icu_ref = (
    spark.table(icustays_table)
    .select("stay_id", "subject_id", "hadm_id", "intime", "outtime",
            F.lit(True).alias("_stay_match"))
)
item_ref = spark.table(d_items_table).select("itemid", F.col("linksto").alias("_linksto"))
# caregiver_id is a string in Silver caregiver; cast to INT for the comparison
caregiver_ref = (
    spark.table(caregiver_table)
    .select(F.expr("TRY_CAST(trim(caregiver_id) AS INT)").alias("caregiver_id"))
    .where(F.col("caregiver_id").isNotNull())
    .distinct()
    .withColumn("_caregiver_match", F.lit(True))
)

rel = (
    outputevents_bronze.select("stay_id", "subject_id", "hadm_id", "caregiver_id", "itemid", "charttime")
    .join(F.broadcast(icu_ref), ["stay_id", "subject_id", "hadm_id"], "left")
    .join(F.broadcast(item_ref), "itemid", "left")
    .join(F.broadcast(caregiver_ref), "caregiver_id", "left")
    .agg(
        _cnt(F.col("_stay_match").isNull()).alias("rows_stay_not_in_icustays"),
        _cnt(F.col("_linksto").isNull()).alias("rows_itemid_not_in_d_items"),
        _cnt(F.col("_linksto").isNotNull() & (F.col("_linksto") != "outputevents")).alias("rows_itemid_linksto_other_table"),
        _cnt(F.col("caregiver_id").isNotNull() & F.col("_caregiver_match").isNull()).alias("rows_caregiver_not_in_caregiver"),
        _cnt(F.col("charttime") < F.col("intime")).alias("rows_charttime_before_intime"),
        _cnt(F.col("charttime") > F.col("outtime")).alias("rows_charttime_after_outtime"),
    )
    .first().asDict()
)
for k, v in rel.items():
    print(f"{k}: {v}")

rows_stay_not_in_icustays: 0
rows_itemid_not_in_d_items: 0
rows_itemid_linksto_other_table: 0
rows_caregiver_not_in_caregiver: 0
rows_charttime_before_intime: 14495
rows_charttime_after_outtime: 1873


## Profiling Findings

Recorded from the profiling run of 2026-10-19/20 on `cdr_mimiciv.bronze.outputevents`
(saved outputs of the previous notebook version). Rows marked *not yet profiled* must be
filled from the cells above before `profiling_approved` is set.
[DOC] = stated in official MIMIC documentation; [OURS] = our engineering choice.

| Check | Result | Silver treatment |
|---|---:|---|
| Rows | 5,359,395 | Row count preserved; [DOC] v2.2 doc lists 4,234,967 (earlier release) |
| Unique row key | None documented | No key asserted; no deduplication |
| Exact duplicate rows | *not yet profiled* | Report only; never removed [OURS] |
| Type casts of Bronze columns | 0 failures | Bronze types passed through; re-cast removed [OURS] |
| NULL in every column | 0 | None |
| `value` < 0 | 604 | `dq_negative_value` [OURS]; value kept |
| `value` NaN | 0 | None |
| `value` range | -3,765 to 4,323,410 | Kept; per-item plausibility decided after the item profile [OURS] |
| `valueuom` values | *not yet profiled* | No change expected |
| `storetime` before `charttime` | 757,757 (14.1%) | `dq_storetime_before_charttime` [OURS]; informational, no correction |
| Gap over 1 h / over 24 h | *not yet profiled* | Informational |
| `charttime` / `storetime` range | 2110-01-11 12:30 to 2214-07-26 14:00 / 2110-01-11 12:50 to 2214-07-26 14:39 | None; [DOC] dates shifted |
| `stay_id` not in icustays / identity mismatch | 0 / 0 | None |
| `itemid` not in `d_items` / `linksto` not outputevents | 0 / 0 | None |
| `caregiver_id` not in caregiver | 0 | None |
| `charttime` outside ICU `intime`/`outtime` | *not yet profiled* | Report only; window filtering is a Gold choice [OURS] |
| Items with negatives / extreme maxima | *not yet profiled* (item diagnostics never ran) | Decide after profiling |

In [0]:
# Silver transformation: all source columns unchanged + DQ flags
# (our engineering choices, see Profiling Findings); no rows removed, no source value changed
outputevents_silver = outputevents_bronze.select(
    *SOURCE_COLUMNS,
    F.coalesce(F.col("value") < 0, F.lit(False))
        .alias("dq_negative_value"),
    F.coalesce(F.col("storetime") < F.col("charttime"), F.lit(False))
        .alias("dq_storetime_before_charttime"),
    F.current_timestamp().alias("_silver_processed_at"),
)

expected_schema = {f.name: f.dataType.simpleString() for f in outputevents_bronze.schema.fields}
expected_schema.update({
    "dq_negative_value": "boolean",
    "dq_storetime_before_charttime": "boolean",
    "_silver_processed_at": "timestamp",
})

# profiled expected counts (2026-10-19/20 run)
PROFILED_COUNTS = {
    "silver_rows": 5359395,
    "dq_negative_value": 604,
    "dq_storetime_before_charttime": 757757,
}

outputevents_silver.printSchema()

root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- stay_id: integer (nullable = true)
 |-- caregiver_id: integer (nullable = true)
 |-- charttime: timestamp (nullable = true)
 |-- storetime: timestamp (nullable = true)
 |-- itemid: integer (nullable = true)
 |-- value: double (nullable = true)
 |-- valueuom: string (nullable = true)
 |-- dq_negative_value: boolean (nullable = false)
 |-- dq_storetime_before_charttime: boolean (nullable = false)
 |-- _silver_processed_at: timestamp (nullable = false)



In [0]:
# publish gate: nothing below runs until profiling has been reviewed and approved
if not PROFILING_APPROVED:
    raise RuntimeError(
        "outputevents is not approved for publication: run and review the profiling cells, "
        "fill the 'not yet profiled' findings and any justified rules, then set widget "
        "profiling_approved=true."
    )

# cheap pre-publish validation (no data scan)
pre_schema = {f.name: f.dataType.simpleString() for f in outputevents_silver.schema.fields}
assert list(pre_schema.items()) == list(expected_schema.items()), "Unexpected Silver schema"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

(
    outputevents_silver.writeTo(TARGET_TABLE)
    .using("delta")
    .createOrReplace()
)

print(f"Published: {TARGET_TABLE}")

Published: cdr_mimiciv.silver.outputevents


In [0]:
# post-publish validation: one aggregate pass over Silver, one count of Bronze (no exceptAll)
published = spark.table(TARGET_TABLE)

post_exprs = [
    F.count("*").alias("silver_rows"),
    _cnt(F.col("_silver_processed_at").isNull()).alias("null_processed_at"),
    _cnt(F.col("dq_negative_value").isNull() | F.col("dq_storetime_before_charttime").isNull())
        .alias("null_dq_flags"),
    _cnt(F.col("dq_negative_value")).alias("dq_negative_value"),
    _cnt(F.col("dq_storetime_before_charttime")).alias("dq_storetime_before_charttime"),
    _cnt(
        (F.col("dq_negative_value") != F.coalesce(F.col("value") < 0, F.lit(False)))
        | (F.col("dq_storetime_before_charttime")
           != F.coalesce(F.col("storetime") < F.col("charttime"), F.lit(False)))
    ).alias("flag_mismatches"),
]
post_exprs += [_cnt(F.col(c).isNull()).alias(f"null_{c}") for c in SOURCE_COLUMNS]
post = published.agg(*post_exprs).first().asDict()
bronze_rows = outputevents_bronze.count()

published_schema = {f.name: f.dataType.simpleString() for f in published.schema.fields}

print("Bronze rows:", bronze_rows)
print("Published:", post)

assert post["silver_rows"] == bronze_rows, "Silver row count differs from Bronze"
assert post["null_processed_at"] == 0
assert post["null_dq_flags"] == 0, "DQ flags must be non-null booleans"
assert post["flag_mismatches"] == 0, "DQ flags do not match their definitions"
assert list(published_schema.items()) == list(expected_schema.items()), \
    f"Schema mismatch: {published_schema}"
# published NULL counts equal the Bronze profile (source values passed through)
for c in SOURCE_COLUMNS:
    assert post[f"null_{c}"] == out_profile[f"null_{c}"], f"NULL count changed for {c}"

if ENFORCE_PROFILED_COUNTS:
    for name, expected in PROFILED_COUNTS.items():
        assert post[name] == expected, f"{name}: expected {expected}, got {post[name]}"

dbutils.notebook.exit("outputevents:OK")